<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# FABnet IPv6 Ext Network: Manual Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** Creates a two-node experiment connected via **FABnet IPv6 Ext**, FABRIC's Layer 3 IPv6 service with **external (public internet) access**. This allows your experiment nodes to reach and be reached from external IPv6-capable hosts. IP addresses and routes are configured manually after the slice becomes active.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create FABnet IPv6 Ext networks (`type='IPv6Ext'`) that provide external internet access
2. Request publicly routable IPv6 addresses with `make_ip_publicly_routable()`
3. Manually assign public IPv6 addresses and configure routes to external destinations
4. Verify connectivity to other FABRIC nodes **and** to external IPv6 hosts
5. Understand routing precautions to avoid disrupting management network access

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with FABnet basics -- see [FABnet IPv6 Manual](../create_l3network_fabnet_ipv6/create_l3network_fabnet_ipv6_manual.ipynb)

**Tip:** For IPv4 external access, see the [FABnet IPv4 Ext](../create_l3network_fabnet_ipv4ext_manual/create_l3network_fabnet_ipv4ext_manual.ipynb) notebook.

</div>

## Background: FABnet IPv6 with External Access

FABRIC provides **FABnetv6Ext**, an extension of the private FABnet IPv6 service that adds routes to the public IPv6 internet. This enables experiments to:

- **Send traffic** to external IPv6-capable hosts
- **Receive traffic** from the public internet on publicly routable IPv6 addresses


<div class="fab-danger">

**Important -- Do NOT set external routes as default!** Adding external routes as the default route will disrupt the management network, and you will lose SSH access to your VM. Only add specific subnet routes to external destinations. If you accidentally misconfigure routes, use `node.os_reboot()` to revert all configuration.

</div>

**NIC component model options:**

| Model | Speed | Type | Ports |
|-------|-------|------|-------|
| `NIC_Basic` | 100 Gbps | Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps | Dedicated Mellanox ConnectX-5 | 2 |
| `NIC_ConnectX_6` | 100 Gbps | Dedicated Mellanox ConnectX-6 | 2 |

## What We're Building

In this notebook we will create two nodes on different sites connected via FABNetv6Ext with external internet access.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We define the external network subnet we want to reach. In this example, we use a Caltech host's IPv6 address. Replace this with the external destination relevant to your experiment.

In [ ]:
# Name for this experiment slice
slice_name = 'MySlice'

# Pick two distinct random FABRIC sites
[site1, site2] = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node names
node1_name = 'Node1'
node2_name = 'Node2'

# Network names -- one per site
network1_name = 'net1'
network2_name = 'net2'

# NIC names
node1_nic_name = 'nic1'
node2_nic_name = 'nic2'

# External network to connect to
# This is an example subnet for a Caltech host -- replace with your target
external_network_subnet = '2605:d9c0:2:10::2:210/64'

## Step 3: Build and Submit the Slice

We create two nodes with NICs connected to `IPv6Ext` networks. No IPs are assigned yet -- we will request public IPs and configure them after the slice is active.

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Node1 (site1) ---
node1 = slice.add_node(name=node1_name, site=site1)
iface1 = node1.add_component(model='NIC_Basic', name=node1_nic_name).get_interfaces()[0]

# --- Node2 (site2) ---
node2 = slice.add_node(name=node2_name, site=site2)
iface2 = node2.add_component(model='NIC_Basic', name=node2_nic_name).get_interfaces()[0]

# --- Networks: IPv6Ext type enables external access ---
net1 = slice.add_l3network(name=network1_name, interfaces=[iface1], type='IPv6Ext')
net2 = slice.add_l3network(name=network2_name, interfaces=[iface2], type='IPv6Ext')

# Submit the slice -- blocks until provisioning is complete (~2-5 min)
slice.submit();

---

## Step 4: Query the Assigned Subnets

FABnet IPv6 Ext networks are assigned a subnet and gateway by FABRIC. Let's query the available IPs.

In [ ]:
# Retrieve the slice and get network information
slice = fablib.get_slice(name=slice_name)

network1 = slice.get_network(name=network1_name)
network1_available_ips = network1.get_available_ips()
network1.show()

network2 = slice.get_network(name=network2_name)
network2_available_ips = network2.get_available_ips()
network2.show();

## Step 5: Request External (Public) Access

You must explicitly request which IPv6 addresses should be publicly routable.

In [ ]:
try:
    # Request public routing for the first available IP on each network
    network1.make_ip_publicly_routable(ipv6=[str(network1_available_ips[0])])

    network2.make_ip_publicly_routable(ipv6=[str(network2_available_ips[0])])

    # Resubmit the slice to apply the public routing changes
    slice.submit()

except Exception as e:
    print(f"Exception: {e}")
    import traceback
    traceback.print_exc()

## Step 6: Configure Node1

Assign the public IPv6 address to Node1's interface and add routes to:
1. The **other FABnet subnet** (Node2's site) via the local gateway
2. An **external IPv6 network** (the Caltech host in this example) via the local gateway

<div class="fab-danger">

**Warning:** Be careful when configuring routes to external networks. Do **not** make these routes default to avoid losing connectivity to management network destinations.

</div>

In [ ]:
# Get the Node1 object and its interface on network1
node1 = slice.get_node(name=node1_name)
node1_iface = node1.get_interface(network_name=network1_name)

# Use the publicly routable IPv6 address
node1_addr = network1.get_public_ips()[0]
node1_iface.ip_addr_add(addr=node1_addr, subnet=network1.get_subnet())

# Route to reach Node2's subnet via our local gateway
node1.ip_route_add(subnet=network2.get_subnet(), gateway=network1.get_gateway())

# Route to reach the external network (Caltech host in this example)
# IMPORTANT: This is a specific route, NOT a default route!
stdout, stderr = node1.execute(f'ip route add {external_network_subnet} via {network1.get_gateway()} dev {node1_iface.get_device_name()}')

# Verify: show the interface configuration and IPv6 routing table
stdout, stderr = node1.execute(f'ip addr show {node1_iface.get_device_name()}')
stdout, stderr = node1.execute(f'ip -6 route list')

## Step 7: Configure Node2

Repeat the same configuration for Node2.

In [ ]:
# Get the Node2 object and its interface on network2
node2 = slice.get_node(name=node2_name)
node2_iface = node2.get_interface(network_name=network2_name)

# Use the publicly routable IPv6 address
node2_addr = network2.get_public_ips()[0]
node2_iface.ip_addr_add(addr=node2_addr, subnet=network2.get_subnet())

# Route to reach Node1's subnet via our local gateway
node2.ip_route_add(subnet=network1.get_subnet(), gateway=network2.get_gateway())

# Route to reach the external network
stdout, stderr = node2.execute(f'ip route add {external_network_subnet} via {network2.get_gateway()} dev {node2_iface.get_device_name()}')

# Verify: show the interface configuration and IPv6 routing table
stdout, stderr = node2.execute(f'ip addr show {node2_iface.get_device_name()}')
stdout, stderr = node2.execute(f'ip -6 route list')

---

## Step 8: Run the Experiment

We test both internal and external connectivity:
1. **Internal:** Ping Node2 from Node1 across the FABRIC backbone
2. **External:** Ping an external IPv6 host (bing.com) from both nodes

In [ ]:
# Retrieve the slice (useful if returning later)
slice = fablib.get_slice(slice_name)

node1 = slice.get_node(name=node1_name)
node2 = slice.get_node(name=node2_name)

node2_addr = node2.get_interface(network_name=network2_name).get_ip_addr()

# Test 1: Internal connectivity -- ping Node2 from Node1
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

# Test 2: External IPv6 connectivity from Node1
stdout, stderr = node1.execute(f'sudo ping -c 5 -I {node1_iface.get_device_name()} bing.com')

# Test 3: External IPv6 connectivity from Node2
stdout, stderr = node1.execute(f'sudo ping -c 5 -I {node2_iface.get_device_name()} bing.com')

## Step 9: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. Public IPv6 addresses and routing resources should be released promptly.

</div>

In [ ]:
# Delete the slice and release all resources
slice = fablib.get_slice(name=slice_name)
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `make_ip_publicly_routable()` throws error | No public IPs available at the site | Try a different site |
| Lost SSH access after adding routes | External route set as default route | Use `node.os_reboot()` to revert configuration |
| Cannot ping external IPv6 hosts | Route not added for that subnet | Add a specific route to the target subnet |
| `ping6` fails to external host | External host not IPv6-capable | Verify the target supports IPv6 |
| Internal ping works but external fails | External routing not configured | Check `ip -6 route list` on the node |
| `PDP Authorization check failed` | Project permissions issue | Contact your project lead or FABRIC support |

## FABlib API Reference

| Method | Description | Documentation |
|--------|-------------|---------------|
| `slice.add_l3network(name, interfaces, type)` | Add a Layer 3 network (`type='IPv6Ext'`) | [add_l3network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l3network) |
| `network.get_available_ips()` | List available IPs on the network | [get_available_ips](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_available_ips) |
| `network.make_ip_publicly_routable(ipv6)` | Request public routing for specified IPv6 addresses | [make_ip_publicly_routable](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.make_ip_publicly_routable) |
| `network.get_public_ips()` | Get the list of publicly routable IPs | [get_public_ips](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_public_ips) |
| `iface.ip_addr_add(addr, subnet)` | Assign an IP address to an interface | [ip_addr_add](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.ip_addr_add) |
| `node.ip_route_add(subnet, gateway)` | Add a static route on the node | [ip_route_add](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.ip_route_add) |
| `node.os_reboot()` | Reboot the node OS (reverts manual config) | [os_reboot](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.os_reboot) |
| `iface.get_device_name()` | Get the Linux device name of the interface | [get_device_name](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_device_name) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **FABnet IPv4 Ext** | [ipv4_ext](../create_l3network_fabnet_ipv4ext_manual/create_l3network_fabnet_ipv4ext_manual.ipynb) | External access with IPv4 addressing |
| **FABnet IPv6 Manual** | [ipv6_manual](../create_l3network_fabnet_ipv6/create_l3network_fabnet_ipv6_manual.ipynb) | Internal-only IPv6 with manual configuration |
| **FABnet IPv6 Auto** | [ipv6_auto](../create_l3network_fabnet_ipv6/create_l3network_fabnet_ipv6_auto.ipynb) | Internal IPv6 with automatic configuration |
| **Facility Ports** | [facility_port](../facility_port/facility_port.ipynb) | Connect FABRIC to external facilities via dedicated links |
| **Port Mirroring** | [port_mirror](../create_port_mirror/port_mirror.ipynb) | Monitor dataplane traffic across slices |